# Profiling ANP - Entendendo o dataset e suas propriedades

In [48]:
import polars as pl

df = pl.read_csv(
    "../data/bronze/anp/automotivos/ano=2023/semestre=1/AUTOMOTIVOS_2023.01.csv",
    separator=";"
    )

linhas, colunas = df.shape

print("\nRegistros:", linhas)
print("\nColunas:", colunas)
print("\nEsquema:", df.schema)

print("\nQuantidade de registros nulos:")

for coluna in df.columns:
    quantidade_nulos = df[coluna].null_count()

    if quantidade_nulos > 0:
        print(f"{coluna}: {quantidade_nulos} ({quantidade_nulos / linhas:.2%})")

print("\nQuantidade de valores distintos de produto:")
print(df["Produto"].n_unique())

print("\nQuantidade de valores distintos de Unidade de Medida:")
print(df["Unidade de Medida"].n_unique())

print("\nQuantidade de valores distintos de Bandeira:")
print(df["Bandeira"].n_unique())

intervalo_datas = df.select(
    data_minima = pl.col("Data da Coleta").str.to_date(format="%d/%m/%Y").min(),
    data_maxima = pl.col("Data da Coleta").str.to_date(format="%d/%m/%Y").max()
)

print("\nIntervalo de datas da coleta:")
print(f"{intervalo_datas[0, 'data_minima']} - {intervalo_datas[0, 'data_maxima']}")

# Teste de chave candidata CNPJ + produto + data da coleta

duplicados = (
    df.select(
        cnpj = pl.col("CNPJ da Revenda"),
        produto = pl.col("Produto"),
        data_coleta = pl.col("Data da Coleta").str.to_date(format="%d/%m/%Y")
    )
    .group_by([
        "cnpj",
        "produto",
        "data_coleta"
    ])
    .len()
    .filter(
        pl.col("len") > 1
    )
)

print("\nCombinações duplicadas:", duplicados.height)


Registros: 431576

Colunas: 16

Esquema: Schema({'Regiao - Sigla': String, 'Estado - Sigla': String, 'Municipio': String, 'Revenda': String, 'CNPJ da Revenda': String, 'Nome da Rua': String, 'Numero Rua': String, 'Complemento': String, 'Bairro': String, 'Cep': String, 'Produto': String, 'Data da Coleta': String, 'Valor de Venda': String, 'Valor de Compra': String, 'Unidade de Medida': String, 'Bandeira': String})

Quantidade de registros nulos:
Numero Rua: 107 (0.02%)
Complemento: 333000 (77.16%)
Bairro: 828 (0.19%)
Valor de Compra: 431576 (100.00%)

Quantidade de valores distintos de produto:
6

Quantidade de valores distintos de Unidade de Medida:
2

Quantidade de valores distintos de Bandeira:
46

Intervalo de datas da coleta:
2023-01-02 - 2023-06-30

Combinações duplicadas: 0
